# SIAs: evaluación para desmovilización

Este notebook consulta el ítem productivo, traduce el estado numérico de cada SIA y detecta las SIAs aprobadas cuya fecha de término de ejecución ya se cumplió. Esas SIAs son candidatas para iniciar el proceso de desmovilización.

**Criterio de candidatura:** `estado == 2` (SIA aprobada), `termino_de_ejecucion` informado y fecha de término menor o igual a hoy. El notebook es únicamente de lectura y no actualiza el servicio.


In [3]:
from getpass import getpass
import os

import pandas as pd
from arcgis.gis import GIS
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [4]:
# Configuración del portal y de los ítems
PORTAL_URL = "https://sig.aminerals.cl/portal/"
USERNAME = "admcenop"

ITEM_INGRESO_ID = "62ce0fc77423403eaf9850efe436506f"
ITEM_PRODUCTIVO_ID = "f6b37b084412444c9d09468031610cb9"
ITEM_DUENIO_AREA_ID = "2300a510323c45c3ad80d1138b6ad50e"
ITEM_INTERSECT_ID = "995f3b915ab84feda3c9b601bd60f8f0"
ITEM_CSV_DUENIO_ID = "9c3770c9ab234958a8c0c6a49f4c22cf"
ITEM_ASCEECC_ID = "f3eeebd77e0b4e47a413b28a6160c943"
ITEM_TABLA_FLUJO = "a1757a4fe46c4f729696ef1e63522b82"

# Defina ARCGIS_PASSWORD en el entorno para evitar escribir la contraseña.
password = os.getenv("ARCGIS_PASSWORD") or getpass("Contraseña de ArcGIS Enterprise: ")
gis = GIS(PORTAL_URL, USERNAME, password)
print(f"Conectado a {gis.properties.portalName} como {gis.users.me.username}")

Conectado a ArcGIS Enterprise como admcenop


## Consulta del ítem productivo

Se consultan todos los atributos, pero no la geometría, porque la evaluación de estados no la necesita.


In [5]:
item_productivo = gis.content.get(ITEM_PRODUCTIVO_ID)
if item_productivo is None:
    raise ValueError(f"No se encontró el ítem productivo {ITEM_PRODUCTIVO_ID}")
if not item_productivo.layers:
    raise ValueError("El ítem productivo no contiene capas consultables")

capa_sias = item_productivo.layers[0]
resultado = capa_sias.query(where="1=1", out_fields="*", return_geometry=False)
df_sias = resultado.sdf.copy()
print(f"Ítem: {item_productivo.title}")
print(f"Capa: {capa_sias.properties.name}")
print(f"Registros consultados: {len(df_sias):,}")

Ítem: CL CEN SIAS - Aprobación
Capa: Solicitud_intervencion_de_areas
Registros consultados: 475


## Interpretación de estados

La siguiente equivalencia reproduce la lógica Arcade entregada para el campo numérico `estado`.


In [6]:
ESTADOS_SIA = {
    0: "Rechazada",
    1: "Pendiente Toma de Conocimiento",
    2: "SIA Aprobada",
    3: "Pendiente Aprobación Gerencia Medioambiente",
    5: "Pendiente Aprobación Dueño/as de Área",
    6: "Pendiente Aprobación Gestor/a Territorial",
    7: "SIA Desmovilizada",
}

df_sias["estado_nombre"] = (
    pd.to_numeric(df_sias["estado"], errors="coerce")
    .astype("Int64")
    .map(ESTADOS_SIA)
    .fillna("Estado desconocido")
)
resumen_estados = (
    df_sias.groupby(["estado", "estado_nombre"], dropna=False).size()
    .rename("cantidad").reset_index().sort_values("estado", na_position="last")
)
display(resumen_estados)

,estado,estado_nombre,cantidad
0,0,Rechazada,13
1,1,Pendiente Toma de Conocimiento,3
2,2,SIA Aprobada,431
3,3,Pendiente Aprobación Gerencia Medioambiente,13
4,5,Pendiente Aprobación Dueño/as de Área,12
5,6,Pendiente Aprobación Gestor/a Territorial,3


## SIAs terminadas que deben pasar a desmovilización


In [7]:
COLUMNAS_FECHA = [
    "fecha", "inicio_de_contrato", "inicio_de_ejecucion",
    "termino_de_ejecucion", "termino_de_desmovilizacion", "termino_de_contrato",
]
for columna in COLUMNAS_FECHA:
    if columna in df_sias.columns:
        df_sias[columna] = pd.to_datetime(df_sias[columna], errors="coerce")

hoy = pd.Timestamp.now().normalize()
es_aprobada = pd.to_numeric(df_sias["estado"], errors="coerce").eq(2)
tiene_termino = df_sias["termino_de_ejecucion"].notna()
ejecucion_terminada = df_sias["termino_de_ejecucion"].dt.normalize().le(hoy)

df_para_desmovilizacion = (
    df_sias.loc[es_aprobada & tiene_termino & ejecucion_terminada]
    .copy().sort_values(["termino_de_ejecucion", "id_sias"])
)
df_para_desmovilizacion["dias_desde_termino"] = (
    hoy - df_para_desmovilizacion["termino_de_ejecucion"].dt.normalize()
).dt.days

columnas_resultado = [
    "id_sias", "nomb_area", "empresa", "nomb_contrato",
    "estado", "estado_nombre", "inicio_de_ejecucion",
    "termino_de_ejecucion", "dias_desde_termino", "termino_de_desmovilizacion",
]
columnas_resultado = [c for c in columnas_resultado if c in df_para_desmovilizacion.columns]
print(f"Fecha de evaluación: {hoy:%Y-%m-%d}")
print(f"SIAs candidatas para desmovilización: {len(df_para_desmovilizacion):,}")
display(df_para_desmovilizacion[columnas_resultado])

Fecha de evaluación: 2026-09-23
SIAs candidatas para desmovilización: 49


,id_sias,nomb_area,empresa,nomb_contrato,estado,estado_nombre,inicio_de_ejecucion,termino_de_ejecucion,dias_desde_termino,termino_de_desmovilizacion
207,000310,Sector de acopio materiales y residuos peligrosos,Minera Centinela,<NA>,2,SIA Aprobada,2025-08-14 16:00:00,2025-09-13 15:00:00,375,2025-09-14 15:00:00
204,000311,Ladera frente a Instalación Chacabuco,MINERA CENTINELA,Implementación Fibra Optica GNSS ESS,2,SIA Aprobada,2025-08-25 16:00:00,2025-10-30 15:00:00,328,2025-10-31 15:00:00
202,000312,Ladera frente a Instalacion Chacabuco,MINERA CENTINELA,Implementación Fibra Optica GNSS ESS,2,SIA Aprobada,2025-08-25 16:00:00,2025-10-30 15:00:00,328,2025-10-31 15:00:00
156,000332,Ruta B-229,Minera Centinela,<NA>,2,SIA Aprobada,2025-10-18 15:00:00,2025-10-31 15:00:00,327,2025-12-01 15:00:00
167,000336,FS - Transporte relaves desde NCEN a DRE Esp,Centinela,<NA>,2,SIA Aprobada,2025-11-05 15:00:00,2025-11-21 15:00:00,306,2026-02-16 15:00:00
210,000343,FS Ductos transporte de relave desde NCEN,Centinela,<NA>,2,SIA Aprobada,2025-11-19 15:00:00,2025-12-14 15:00:00,283,2026-02-25 15:00:00
147,000335,Complemento área GS,Minera Centinela,<NA>,2,SIA Aprobada,2025-10-30 15:00:00,2025-12-20 15:00:00,277,2025-12-31 15:00:00
135,000344,Calicatas para análisis ductos transporte relaves desde NCEN,Centinela,<NA>,2,SIA Aprobada,2025-11-24 15:00:00,2025-12-22 15:00:00,275,2026-02-23 15:00:00
201,000315,Ground Station DES,Centinela,<NA>,2,SIA Aprobada,2025-09-08 15:00:00,2025-12-31 15:00:00,266,2026-01-31 15:00:00
194,000323,Complemento área GS,Minera Centinela,<NA>,2,SIA Aprobada,2025-09-10 15:00:00,2025-12-31 15:00:00,266,2026-01-05 15:00:00


In [10]:
for c in df_para_desmovilizacion['id_sias']:
   print(c)

000310
000311
000312
000332
000336
000343
000335
000344
000315
000323
000331
000338
000340
000324
000342
000359
000330
000366
000372
000362
000353
000320
000364
000374
000412
000380
000403
000375
000414
000385
000348
000420
000426
000329
000373
000379
000369
000370
000387
000382
000454
000384
000341
000446
000447
000413
000433
000463
000489


In [12]:
df_para_desmovilizacion['id_sias'].max()

'000489'

In [13]:
df_para_desmovilizacion.to_excel(r"D:\SIAS-DESM\sias-centinela-desmovilizacion\Excel\SIAs_para_desmovilizacion.xlsx", index=False)

In [14]:
df_para_desmovilizacion.columns

Index(['objectid', 'globalid', 'nombre', 'rut', 'numero', 'email', 'fecha',
       'empresa', 'nomb_contrato', 'gerencias',
       ...
       'numb_contrato_num', 'numb_contrato', 'token_admin', 'obs_duenio',
       'obs_mamb', 'obs_inggter', 'obs_admc_eecc', 'resumen_multida',
       'estado_nombre', 'dias_desde_termino'],
      dtype='str', length=107)

## SIAs ya desmovilizadas y control de calidad


In [8]:
df_desmovilizadas = df_sias.loc[
    pd.to_numeric(df_sias["estado"], errors="coerce").eq(7)
].copy()
descripcion_servicio = df_sias["estado_desc"].fillna("").str.strip()
df_estado_inconsistente = df_sias.loc[
    descripcion_servicio.ne("")
    & descripcion_servicio.str.casefold().ne(df_sias["estado_nombre"].str.casefold()),
    ["id_sias", "estado", "estado_nombre", "estado_desc"],
].copy()
print(f"SIAs ya desmovilizadas (estado 7): {len(df_desmovilizadas):,}")
print(f"Registros con diferencia entre estado y estado_desc: {len(df_estado_inconsistente):,}")
display(df_estado_inconsistente)

SIAs ya desmovilizadas (estado 7): 0
Registros con diferencia entre estado y estado_desc: 6


,id_sias,estado,estado_nombre,estado_desc
1,000511,6,Pendiente Aprobación Gestor/a Territorial,Revision
8,000513,1,Pendiente Toma de Conocimiento,Revision
11,000512,1,Pendiente Toma de Conocimiento,Revision
70,000330,2,SIA Aprobada,Pendiente Toma de Conocimiento
149,000333,3,Pendiente Aprobación Gerencia Medioambiente,Pendiente Toma de Conocimiento
162,000320,2,SIA Aprobada,Pendiente Toma de Conocimiento


In [15]:
sias_desmovilizacion = '000310'

In [ ]:
## Que datos voy a mostrar de desmovilizacion
